<a href="https://colab.research.google.com/github/Chosencodes/Medical-Imaging-Projects/blob/main/Cardiac_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Preprocessing**

In [ ]:
!pip install pydicom

In [ ]:
from pathlib import Path
import pydicom
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge

In [ ]:
!unzip -q rsna-pneumonia-detection-challenge.zip -d /content/rsna

In [ ]:
labels = pd.read_csv("/content/drive/MyDrive/05-Detection/rsna_heart_detection.csv")

In [ ]:
labels.head(3)

In [ ]:
ROOT_PATH = Path("/content/rsna/stage_2_train_images/")
SAVE_PATH = Path("Processed-Heart-Detection")

In [ ]:
import os

all_files = set(os.listdir("/content/rsna/stage_2_train_images/"))

labels = labels[labels["name"].apply(lambda x: x + ".dcm" in all_files)].reset_index(drop=True)

In [ ]:
fig,axis = plt.subplots(2,2,figsize=(10,10))
c = 0

for i in range(2):
  for j in range(2):
    data = labels.iloc[c]
    patient_id = data["name"]
    dcm_path =ROOT_PATH/str(patient_id)
    dcm_path = dcm_path.with_suffix('.dcm')
    dcm = pydicom.dcmread(dcm_path).pixel_array

    dcm_array = cv2.resize(dcm,(224,224))

    x = data["x0"]
    y = data["y0"]
    w = data["w"]
    h = data["h"]

    axis[i][j].imshow(dcm_array,cmap="bone")
    rect = patches.Rectangle((x,y),w,h,linewidth=1,edgecolor="r",facecolor="none")
    axis[i][j].add_patch(rect)
    c+=1



In [ ]:
sums = 0
sums_squared = 0
train_ids = []
val_ids = []

for c, patient_id in enumerate(list(labels.name)):
  dcm_path = ROOT_PATH/str(patient_id)
  dcm_path = dcm_path.with_suffix('.dcm')
  dcm = pydicom.dcmread(dcm_path).pixel_array

  dcm_array = (cv2.resize(dcm,(224,224)) / 255).astype(np.float16)

  train_or_val = "train" if c < 400 else "val"

  if train_or_val == "train":
    train_ids.append(patient_id)
  else:
    val_ids.append(patient_id)

  current_save_path = SAVE_PATH/train_or_val/patient_id
  current_save_path.mkdir(parents=True, exist_ok=True)
  np.save(current_save_path/patient_id,dcm_array)

  normalizer = 224*224
  if train_or_val == "train":
    sums += np.sum(dcm_array) / normalizer
    sums_squared += (dcm_array ** 2).sum() / normalizer

In [ ]:
np.save("Processed-Heart-Detection/train_subjects_det",train_ids)
np.save("Processed-Heart-Detection/val_subjects_det",val_ids)

In [ ]:
mean = sums / len(train_ids)
std = np.sqrt((sums_squared / len(train_ids)) - mean ** 2)

In [ ]:
mean,std

# **DATASET**

In [ ]:
dataset_code = '''
import torch
import numpy as np
import pandas as pd
import albumentations as A
from pathlib import Path
import os

class CardiacDataset(torch.utils.data.Dataset):
  def __init__(self, path_to_labels_csv, patients, root_path, augs=None):
    self.labels = pd.read_csv(path_to_labels_csv)
    self.patients = np.load(patients)
    self.root_path = Path(root_path)
    self.augment = augs

  def __len__(self):
    return len(self.patients)

  def __getitem__(self, idx):
    patient = self.patients[idx]
    data = self.labels[self.labels["name"] == patient]

    x_min = data["x0"].item()
    y_min = data["y0"].item()
    x_max = x_min + data["w"].item()
    y_max = y_min + data["h"].item()

    file_path = self.root_path / patient / f"{patient}.npy"
    img = np.load(str(file_path)).astype(np.float32)

    if self.augment:
        img_uint8 = (img * 255).clip(0, 255).astype(np.uint8)
        img_uint8 = np.expand_dims(img_uint8, axis=-1)

        transformed = self.augment(
            image=img_uint8,
            bboxes=[[x_min, y_min, x_max, y_max]],
            labels=["heart"]
        )

        img = transformed["image"].squeeze(-1).astype(np.float32) / 255.0
        if transformed["bboxes"]:
            x_min, y_min, x_max, y_max = transformed["bboxes"][0]

    img = (img - 0.494) / 0.253
    img = torch.tensor(img).unsqueeze(0)
    bbox = torch.tensor([x_min, y_min, x_max, y_max])

    return img, bbox
'''

with open("/content/dataset.py", "w") as f:
    f.write(dataset_code)

In [ ]:
import sys
if "dataset" in sys.modules: sys.modules.pop("dataset")
from dataset import CardiacDataset

In [ ]:
train_augs = A.Compose([
    A.RandomGamma(p=0.5),
    A.Affine(scale = (0.8, 1.2),rotate = (-10, 10),translate_px = {"x": (-10, 10), "y": (-10, 10)}),
], bbox_params=A.BboxParams(format = "pascal_voc",label_fields = ["labels"], clip = True
))


# **VALIDATE**

In [ ]:
labels_path ="/content/drive/MyDrive/05-Detection/rsna_heart_detection.csv"
patients_path ="/content/drive/MyDrive/05-Detection/train_subjects.npy"
train_root = "Processed-Heart-Detection/train/"

In [ ]:
dataset = CardiacDataset(labels_path,patients_path,train_root,train_augs)

In [ ]:
import shutil
shutil.copy(
    "/content/drive/MyDrive/05-Detection/01-Preprocess.ipynb",
    "/content/01-Preprocess.ipynb"
)
print("done")

In [ ]:

img, bbox = dataset[0]

fig, axis = plt.subplots(1,1)
axis.imshow(img[0],cmap="bone")
rect = patches.Rectangle((bbox[0],bbox[1]), bbox[2]-bbox[0],bbox[3]-bbox[1],edgecolor="r",facecolor="none")
axis.add_patch(rect)

# **Train**

In [ ]:
!pip install pytorch-lightning torchmetrics

In [ ]:
import torch
import torchvision
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from dataset import CardiacDataset

In [ ]:
train_root_path = "Processed-Heart-Detection/train/"
train_subjects ="/content/drive/MyDrive/05-Detection/train_subjects.npy"
val_root_path = "Processed-Heart-Detection/val"
val_subjects = "/content/drive/MyDrive/05-Detection/val_subjects.npy"
labels_path ="/content/drive/MyDrive/05-Detection/rsna_heart_detection.csv"

In [ ]:
train_transforms = A.Compose([
    A.RandomGamma(p=0.5),
    A.Affine(scale = (0.8, 1.2),rotate = (-10, 10),translate_px = {"x": (-10, 10), "y": (-10, 10)}),
], bbox_params=A.BboxParams(format = "pascal_voc",label_fields = ["labels"], clip = True
))


In [ ]:
train_dataset = CardiacDataset(labels_path,train_subjects,train_root_path,augs=train_transforms)
val_dataset = CardiacDataset(labels_path,val_subjects,val_root_path,augs=None)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=8,num_workers=4,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset,batch_size=8,num_workers=4,shuffle=False)

In [ ]:
print(f"There are {len(train_dataset)} train images and {len(val_dataset)} val images")

In [ ]:
# torchvision.models.resnet18()

In [ ]:
class CardiacDetectionModel(pl.LightningModule):
  def __init__(self):
    super().__init__()
    self.model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    self.model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    self.model.fc = torch.nn.Linear(in_features=512, out_features=4)

    self.loss_fn = torch.nn.MSELoss()

  def forward(self,data):
    return self.model(data)

  def training_step(self,batch,batch_idx):
    x_ray,label = batch
    label = label.float()
    pred = self(x_ray)
    loss = self.loss_fn(pred,label)

    self.log("Train loss",loss,prog_bar=True)

    if batch_idx % 50 == 0:
      self.log_images(x_ray.cpu(),pred.cpu(),label.cpu(),"Train")
    return loss

  def validation_step(self,batch,batch_idx):
    x_ray,label = batch
    label = label.float()
    pred = self(x_ray)
    loss = self.loss_fn(pred,label)

    self.log("Val loss",loss,prog_bar=True)

    if batch_idx % 50 == 0:
      self.log_images(x_ray.cpu(),pred.cpu(),label.cpu(),"Val")
    return loss

  def log_images(self,x_ray,pred,label,name):
    results = []

    for i in range(4):
      coords_labels = label[i]
      coords_pred = pred[i]

      img = ((x_ray[i] * 0.253) + 0.494).numpy().squeeze()
      img = np.stack([img, img, img], axis=-1).copy()

      x0,y0 = coords_labels[0].int().item(),coords_labels[1].int().item()
      x1,y1 = coords_labels[2].int().item(),coords_labels[3].int().item()
      img = cv2.rectangle(img,(x0,y0),(x1,y1),(0,0,0),2)

      x0,y0 = coords_pred[0].int().item(),coords_pred[1].int().item()
      x1,y1 = coords_pred[2].int().item(),coords_pred[3].int().item()
      img = cv2.rectangle(img,(x0,y0),(x1,y1),(1,1,1),2)

      results.append(torch.tensor(img).permute(2, 0, 1))

    grid = torchvision.utils.make_grid(results,nrow=2)
    self.logger.experiment.add_image(name,grid,self.global_step)

  def configure_optimizers(self):
      optimizer = torch.optim.Adam(self.parameters(),lr=1e-4)
      return optimizer


In [ ]:
model = CardiacDetectionModel()
print(hasattr(model, 'configure_optimizers'))

In [ ]:
checkpoint_callback = ModelCheckpoint(
    filename="best-checkpoint",
    save_top_k=10,
    monitor="Val loss",
    mode="min"
)

In [ ]:
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=100,
    logger=TensorBoardLogger(save_dir="/content/logs"),
    log_every_n_steps=1,
    callbacks=[checkpoint_callback]

)

In [ ]:
trainer.fit(model,train_loader,val_loader)

# **Evaluation**

In [ ]:
import os
log_path = "/content/drive/MyDrive/05-Detection/"
for root, dirs, files in os.walk(log_path):
    for f in files:
        if f.endswith(".ckpt"):
            print(os.path.join(root, f))

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = CardiacDetectionModel.load_from_checkpoint("/content/drive/MyDrive/05-Detection/weight.ckpt")
model = model.to(device)
model.eval()